In [43]:
from langchain_core.tools import tool
import requests
from langchain_community.tools import DuckDuckGoSearchRun

In [44]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq

load_dotenv()

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

In [45]:
search_tool = DuckDuckGoSearchRun()
@tool
def get_weather_data(city: str) -> str:
  """
  This function fetches the current weather data for a given city
  """
  url = f'https://api.weatherstack.com/current?access_key=4d1d8ae207a8c845a52df8a67bf3623e&query={city}'

  response = requests.get(url)

  return response.json()

In [46]:
import langchain.agents as agents

print(dir(agents))

['AgentState', '__all__', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', 'create_agent', 'factory', 'middleware', 'structured_output']


In [47]:
from langchain_classic.agents import AgentExecutor
from langchain_classic.agents import create_react_agent
from langsmith import Client

client = Client()

prompt = client.pull_prompt(
    "hwchase17/react",
    dangerously_pull_public_prompt=True
)

In [48]:
# Create the ReAct agent manually with the pulled prompt
agent = create_react_agent(
    llm=llm,
    tools=[search_tool, get_weather_data],
    prompt=prompt
)

In [49]:
# Wrap it with AgentExecutor
agent_executor = AgentExecutor(
    agent=agent,
    tools=[search_tool, get_weather_data],
    verbose=True
)

In [50]:
response = agent_executor.invoke({"input": "Find the capital of Madhya Pradesh, then find it's current weather condition"})
print(response)



> Entering new AgentExecutor chain...
Thought: To find the capital of Madhya Pradesh, I can use my general knowledge or search for it. Since I'm not sure about the current weather condition, I'll need to use the get_weather_data function after finding the capital.

Action: duckduckgo_search
Action Input: capital of Madhya PradeshIndia is a federal union comprising 28 states and 8 union territories, for a total of 36 subnational entities. Nagaland became a state in 1963, with Kohima as its capital. Madhya Pradesh and Chhattisgarh were two other states formed after India’s independence, with Bhopal and Raipur as their capitals, respectively. While most know it as the capital city of Madhya Pradesh, Bhopal district is much more — a complete destination that invites tourists of every kind. Whether you're a history enthusiast, a nature lover, a foodie, or a culture buff, Bhopal has a story for you. ° to 86°). Madhya Pradesh receives an average annual rainfall of about 1200 mm (nearly 50 i

In [51]:
response['output']

"The capital of Madhya Pradesh is Bhopal, but unfortunately, I couldn't retrieve the current weather condition due to usage limit restrictions."